### Chapter 5.7: MediaPipe BlazePose - 실시간 자세 추정
### 의료 현장 움직임 분석

### Chapter 5.7: MediaPipe BlazePose 자세 추정
#
### 실시간 보행 분석 및 임상 평가

##### 1. 라이브러리 설치


In [ ]:
# !pip install mediapipe opencv-python numpy matplotlib


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

try:
    import mediapipe as mp
    print("✅ MediaPipe 설치 확인")
except ImportError:
    print("❌ MediaPipe가 설치되지 않음")
    print("설치: pip install mediapipe")

##### 2. MediaPipe BlazePose 초기화


In [ ]:
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# Pose 모델 초기화
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=2,          # 0, 1, 2 (높을수록 정확)
    smooth_landmarks=True,       # 랜드마크 스무딩
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

print("✅ BlazePose 초기화 완료")

##### 3. 웹캠 실시간 자세 추정


In [ ]:
# 웹캠 열기
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("❌ 카메라를 열 수 없습니다.")
else:
    print("📹 웹캠 시작 (q 키로 종료)")

    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        frame_count += 1

        # BGR → RGB 변환
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 자세 추정
        results = pose.process(frame_rgb)

        # 랜드마크 그리기
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style()
            )

        # 프레임 카운터 표시
        cv2.putText(
            frame,
            f'Frame: {frame_count}',
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        # 화면 표시
        cv2.imshow('BlazePose - Press Q to quit', frame)

        # 'q' 키로 종료
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    pose.close()

    print(f"✅ 총 {frame_count} 프레임 처리")

##### 4. 비디오 파일 자세 추정


In [ ]:
# 비디오 파일 경로 (실제 파일로 변경 필요)
VIDEO_PATH = 'sample_video.mp4'

# 비디오 파일 존재 확인
import os
if os.path.exists(VIDEO_PATH):
    print(f"📹 비디오 파일: {VIDEO_PATH}")

    cap = cv2.VideoCapture(VIDEO_PATH)

    # 비디오 정보
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"  - 해상도: {width}x{height}")
    print(f"  - FPS: {fps}")
    print(f"  - 총 프레임: {total_frames}")

    # 출력 비디오 설정
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter('output_pose.mp4', fourcc, fps, (width, height))

    # Pose 초기화
    pose = mp_pose.Pose()

    keypoints_history = []
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # BGR → RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 자세 추정
        results = pose.process(frame_rgb)

        # 키포인트 저장
        if results.pose_landmarks:
            keypoints = []
            for landmark in results.pose_landmarks.landmark:
                keypoints.append({
                    'x': landmark.x,
                    'y': landmark.y,
                    'z': landmark.z,
                    'visibility': landmark.visibility
                })
            keypoints_history.append(keypoints)

        # 랜드마크 그리기
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS
            )

        # 진행률 표시
        cv2.putText(
            frame,
            f'Frame: {frame_count}/{total_frames}',
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        # 비디오 저장
        out.write(frame)

        # 화면 표시 (선택)
        cv2.imshow('Processing...', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    pose.close()

    print(f"✅ 처리 완료: output_pose.mp4")
    print(f"  - 처리된 프레임: {frame_count}")
    print(f"  - 키포인트 추출: {len(keypoints_history)} 프레임")

else:
    print(f"⚠️  비디오 파일이 없습니다: {VIDEO_PATH}")
    keypoints_history = []

##### 5. 관절 각도 계산


In [ ]:
# 3개 점으로 관절 각도 계산
def calculate_angle(p1, p2, p3):
    """p2를 중심으로 하는 각도"""
    v1 = np.array([p1['x'] - p2['x'], p1['y'] - p2['y']])
    v2 = np.array([p3['x'] - p2['x'], p3['y'] - p2['y']])

    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    angle = np.arccos(np.clip(cos_angle, -1.0, 1.0))

    return np.degrees(angle)

In [ ]:
# 무릎 각도 분석
if keypoints_history:
    left_knee_angles = []
    right_knee_angles = []

    for keypoints in keypoints_history:
        # 좌측 무릎: hip(23) - knee(25) - ankle(27)
        left_angle = calculate_angle(keypoints[23], keypoints[25], keypoints[27])
        left_knee_angles.append(left_angle)

        # 우측 무릎: hip(24) - knee(26) - ankle(28)
        right_angle = calculate_angle(keypoints[24], keypoints[26], keypoints[28])
        right_knee_angles.append(right_angle)

    print(f"✅ 무릎 각도 계산 완료: {len(left_knee_angles)} 프레임")

##### 6. 시간에 따른 관절 각도 그래프


In [ ]:
# 무릎 각도 플롯
if keypoints_history:
    plt.figure(figsize=(12, 6))

    plt.plot(left_knee_angles, label='Left Knee', color='blue', linewidth=2)
    plt.plot(right_knee_angles, label='Right Knee', color='red', linewidth=2)

    plt.title('Knee Angles Over Time', fontsize=16)
    plt.xlabel('Frame', fontsize=12)
    plt.ylabel('Angle (degrees)', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)

    plt.savefig('knee_angles.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("그래프 저장: knee_angles.png")

##### 7. 보행 분석


In [ ]:
# 보폭 계산
if keypoints_history:
    stride_lengths = []

    for i in range(1, len(keypoints_history)):
        prev = keypoints_history[i-1]
        curr = keypoints_history[i]

        # 왼발 발꿈치 이동 거리
        left_foot_prev = prev[29]  # left_heel
        left_foot_curr = curr[29]

        distance = np.sqrt(
            (left_foot_curr['x'] - left_foot_prev['x'])**2 +
            (left_foot_curr['y'] - left_foot_prev['y'])**2
        )

        stride_lengths.append(distance)

    avg_stride = np.mean(stride_lengths)
    print(f"평균 보폭: {avg_stride:.4f} (정규화)")

In [ ]:
# 보행 비대칭 점수
if keypoints_history:
    angle_diff = np.abs(np.array(left_knee_angles) - np.array(right_knee_angles))
    asymmetry_score = np.mean(angle_diff) / 180.0

    print(f"보행 비대칭 점수: {asymmetry_score:.3f}")
    print(f"   (0 = 완전 대칭, 1 = 최대 비대칭)")

##### 8. 임상 분석 요약


In [ ]:
if keypoints_history:
    print("\n" + "="*60)
    print("임상 분석 요약")
    print("="*60)

    print(f"분석 프레임 수: {len(keypoints_history)}")
    print(f"평균 보폭: {avg_stride:.4f}")
    print(f"보행 비대칭: {asymmetry_score:.3f}")

    # 무릎 각도 통계
    print(f"\n무릎 각도 통계:")
    print(f"  좌측 - 평균: {np.mean(left_knee_angles):.1f}°, "
          f"범위: {np.min(left_knee_angles):.1f}° ~ {np.max(left_knee_angles):.1f}°")
    print(f"  우측 - 평균: {np.mean(right_knee_angles):.1f}°, "
          f"범위: {np.min(right_knee_angles):.1f}° ~ {np.max(right_knee_angles):.1f}°")

    print("="*60)

##### 9. 33개 키포인트 위치


In [ ]:
# BlazePose 33개 랜드마크
LANDMARK_NAMES = [
    'nose', 'left_eye_inner', 'left_eye', 'left_eye_outer',
    'right_eye_inner', 'right_eye', 'right_eye_outer',
    'left_ear', 'right_ear', 'mouth_left', 'mouth_right',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_pinky', 'right_pinky',
    'left_index', 'right_index', 'left_thumb', 'right_thumb',
    'left_hip', 'right_hip', 'left_knee', 'right_knee',
    'left_ankle', 'right_ankle', 'left_heel', 'right_heel',
    'left_foot_index', 'right_foot_index'
]

print("BlazePose 33개 키포인트:")
for i, name in enumerate(LANDMARK_NAMES):
    print(f"  {i:2d}. {name}")